In [1]:
## imports
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const


## set unit equialencies
u.set_enabled_equivalencies(u.dimensionless_angles())

The deflection of an apparent star position by a GW passing through Earth is given by ([Book & Flanagan 2011](https://ui.adsabs.harvard.edu/abs/2011PhRvD..83b4024B/abstract)):

$$dn^{i}(t,\vec{n}) = \frac{n^i-q^i}{2\left(1-\vec{q}\cdot\vec{n}\right)}h_{jk}(t,\vec{0})n^jn^k-\frac{1}{2}h^{ij}(t,\vec{0})n_j$$

where $\vec{n}$ is the on-sky star position, $\vec{q}$ is the on-sky GW source position, and $\vec{0}$ is the position of the observer (e.g., Earth). 

This can be reasonably approximated as:

$$dn \sim \frac{1}{2} h \sin^2\theta $$

where $\theta$ is the angular distance between the GW source and a given star. I'm going to ignore the angular part of this in what follows (i.e., set $\theta = \pi/2$).

The strain sensitivity for a given survey is then roughly:
$$ h = \frac{2\sigma_{ss}}{\sqrt{N_{\rm{meas}} N_{\rm{stars}}}}$$

where $\sigma_{ss}$ is the single-image, single-star astrometric precision, $N_{\rm{meas}}$ is the number of measurements of each star, and $N_{\rm{stars}}$ is the number of stars observed. Note that we can rewrite $N_{\rm{meas}} = T_{\rm{survey}}/t_{\rm{cadence}}$, if we assume the survey is continuously sampled at the same cadence. This estimate is encoded below as `h_sens()`.

This estimate can be converted to a PSD via the following equation:
$$P_n(f) = 4 t_{\rm{cadence}} \frac{\sigma^2_{ss}}{N_{\rm{meas}} N_{\rm{stars}}} $$

This is derived in the text above Eqn. 10 of [Wang et al. (2022)](https://ui.adsabs.harvard.edu/abs/2022PhRvD.106h4006W/abstract). I have given the equation with slightly different notation here. This is implemented in `psd_simple()` below. Note that this is independent of frequency, which matches what we see in our data (for white noise only). Also note that this is the two-sided PSD. 

In [2]:
## simple h sens. for astrometry
def h_sens(sig_ss, Tsurvey, tcad, Nstars):
    '''
    all inputs should be astropy quantities with attached units, except Nstars.
    '''
    sig_ss = sig_ss.to(u.rad)
    Tsurvey = Tsurvey.to(u.s)
    tcad = tcad.to(u.s)
    Nmeas = (Tsurvey/tcad).to('')
    return (2. * sig_ss/np.sqrt(Nmeas * Nstars)).to('')

In [3]:
## simple psd for astrometry
def psd_simple(sig_ss, Tsurvey, tcad, Nstars):
    '''
    2-sided PSD
    all inputs should be astropy quantities with attached units, except Nstars.
    '''
    h = h_sens(sig_ss, Tsurvey, tcad, Nstars)
    return (h**2 * tcad).to(1./u.Hz)

For _Kepler_, the values for the above survey parameters are:

$\sigma_{ss} \sim 0.7~\rm{mas}$

$T_{\rm{survey}} \sim 3.5~\rm{years}$

$t_{\rm{cadence}} \sim 30~\rm{minutes}$

$N_{\rm{stars}} \sim 160,000$

Plugging this all in gives:

In [4]:
h_sens(0.7 * u.mas, 3.5 * u.year, 30 * u.min, 160000)

<Quantity 6.8500408e-14>

In [5]:
psd_simple(0.7 * u.mas, 3.5 * u.year, 30 * u.min, 160000)

<Quantity 8.4461506e-24 1 / Hz>

When adjusted for the timescales and star numbers we are using for tests, our results from our actual pipeline are about a factor of 2 worse than the estimates we get here (which can be roughly accounted for by averaging over sky locations). Note that again this is only the case for the frequency range where we have roughly white noise.

If you are interested in doing calculations that involve the position of the _Kepler_ field relative to pulsars, then the FOV center for _Kepler_ is: 

RA = 19h22m40s, DEC = +44:30:00 (ICRS)

It has a FOV size of roughly 10 deg x 10 deg.